<a href="https://colab.research.google.com/github/vad-source/NLPAPP/blob/main/QA/NLPAPP_DeepLearning_Based_Generative_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## NLP APPLICATIONS
**Designed by:** RAJA VADHANA PRABHAKAR  
**Organization:** BITS PILANI WILP  
**Purpose:** Academic Training / Proof of Concept  

---
#### Attribution & AI Disclosure
- **Original Design:** The logic, architecture, and modular structure of this notebook were designed by the author.
- **Development Assistance:** Generative AI (e.g., ChatGPT/Claude/Copilot) was used for coding implementation and debugging support.
- **License:** This work is licensed under the [Apache License 2.0](https://apache.org).

In [ ]:
!pip -q install transformers sentencepiece torch pandas

In [ ]:
import torch
import pandas as pd
from transformers import (AutoTokenizer,AutoModelForSeq2SeqLM)


## 0. Knowledge Base

In [ ]:
documents = [{"id": 1,"context":"""Customer purchased a laptop last week.Battery drains within one hour.Screen flickers intermittently.Customer already reinstalled drivers. Warranty is active for one year."""},
             {"id": 2,"context":"""Customer reports internet disconnects frequently.Router firmware is outdated.Restart temporarily fixes issue.ISP outage was not detected."""}
]

qa_test = [{"question":"What should the customer do next?","context_id": 1,"ground_truth":""" Run hardware diagnostics and contact support for warranty replacement."""},
           {"question":"What is likely causing the internet issue?","context_id": 2,"ground_truth":"""The outdated router firmware is likely causing instability."""}
]

## 1. Query Processor

In [ ]:
#Learners may replace this with sophisticated techiques like GEC Question restructuting etc.,
class QueryProcessor:
    def process(self, question):
        return question.strip()


In [ ]:
query_processorT = QueryProcessor()
context_idT = 1
processed_questionT = query_processorT.process(qa_test[context_idT]["question"])
print("Sample Query : ", qa_test[context_idT]["question"])
print("Processed Query: ", processed_questionT)

Sample Query :  What is likely causing the internet issue?
Processed Query:  What is likely causing the internet issue?


In [ ]:
#Learners may replace this with sophisticated techiques like automated intent/context extraction
class DocumentProcessor:
    def get_context(self, context_id):
        for doc in documents:
            if doc["id"] == context_id:
                return doc["context"]
        return ""


In [ ]:
#This is just added to illustrate the internal working of sub word tokenization performed by BERT.
#This code is not used elsewhere in this code file.
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
text = "unbelievable NLP systems"
tokens = tokenizer.tokenize(text)
print(tokens)

['▁unbelievable', '▁N', 'LP', '▁systems']


## 2. Candidate Retriever

In [ ]:
class GenerativeQA:
    def __init__(self):
        print("Loading FLAN-T5 model...")
        self.model_name = "google/flan-t5-small"
        self.tokenizer =AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)

    def generate_answer(self,question,context):
        prompt = f"""Answer the question based on context. Context:{context} Question: {question} Answer:"""
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        outputs = self.model.generate(**inputs,max_new_tokens=64, do_sample=True,temperature=0.3,top_p=0.9)
        answer = self.tokenizer.decode(outputs[0],skip_special_tokens=True)
        return answer

## 3. Answer Generator

In [ ]:
#Learners may use this section to code automated modification/rewrite/reordering/filtering of candidate answers before final answers gets curated
class PostProcessor:
    def curate(self, answer):
        return answer.strip()

## 4. Evaluator

In [ ]:
class Evaluator:
    def exact_match(self, pred, gt):
        return int(pred.lower().strip()==gt.lower().strip())

    def token_f1(self, pred, gt):
        pred_tokens = pred.lower().split()
        gt_tokens = gt.lower().split()
        common = set(pred_tokens) & set(gt_tokens)
        if len(common) == 0:
            return 0
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(gt_tokens)
        f1 = (2 * precision * recall) / (precision + recall)
        return round(f1, 3)

    def semantic_overlap(self, pred, gt):
        pred_words = set(pred.lower().split())
        gt_words = set(gt.lower().split())
        overlap = len(pred_words & gt_words)
        union = len(pred_words | gt_words)
        return round(overlap / union, 3)

    def aggregate_score(self, em, f1, semantic):
        return round((em + f1 + semantic) / 3, 3)


##

## Pipeline

In [ ]:
print("=" * 60)
print("GENERATIVE QA SYSTEM")
print("=" * 60)

query_processor = QueryProcessor()
doc_processor = DocumentProcessor()
gen_qa = GenerativeQA()
postprocessor = PostProcessor()
evaluator = Evaluator()

all_scores = []
results = []
for sample in qa_test:
    print("\n" + "=" * 60)
    question = sample["question"]
    context_id = sample["context_id"]
    gt = sample["ground_truth"]
    print("\nQUESTION:")
    print(question)
    print("\n[PHASE 1] QUERY PROCESSING")
    processed_question = query_processor.process(question)
    print("\n[PHASE 2] DOCUMENT PROCESSING")
    context = doc_processor.get_context(context_id)
    print("\nCONTEXT:")
    print(context)
    print("\n[PHASE 3] GENERATIVE QA")
    raw_answer = gen_qa.generate_answer(processed_question, context)
    print("\nRAW GENERATED ANSWER:")
    print(raw_answer)
    print("\n[PHASE 4] CANDIDATE GENERATION")
    candidate_answer = raw_answer
    print("\n[PHASE 5] POSTPROCESSING")
    final_answer = postprocessor.curate(candidate_answer)
    print("\nFINAL ANSWER:")
    print(final_answer)
    print("\n[PHASE 6] EVALUATION")
    em = evaluator.exact_match(final_answer,gt)
    f1 = evaluator.token_f1(final_answer,gt)
    semantic = evaluator.semantic_overlap(final_answer,gt)
    final_score = evaluator.aggregate_score(em,f1,semantic)
    print("\nGROUND TRUTH:")
    print(gt)
    print("\nMETRICS")
    print("Exact Match:", em)
    print("Token F1:", f1)
    print("Semantic Overlap:", semantic)
    print("Aggregate Score:",final_score)
    all_scores.append(final_score)
    results.append({"Question": question,"Predicted": final_answer,"Ground Truth": gt,"Exact Match": em,"Token F1": f1,"Semantic Overlap": semantic,"Final Score": final_score})

print("\n" + "=" * 60)
print("FINAL AGGREGATED RESULTS")
print("=" * 60)
avg_score = sum(all_scores) / len(all_scores)
print("\nAverage QA Score:", round(avg_score, 3))

df = pd.DataFrame(results)
print("\nDETAILED RESULTS")
display(df)

GENERATIVE QA SYSTEM
Loading FLAN-T5 model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]



QUESTION:
What should the customer do next?

[PHASE 1] QUERY PROCESSING

[PHASE 2] DOCUMENT PROCESSING

CONTEXT:
Customer purchased a laptop last week.Battery drains within one hour.Screen flickers intermittently.Customer already reinstalled drivers. Warranty is active for one year.

[PHASE 3] GENERATIVE QA

RAW GENERATED ANSWER:
Update laptop and other driver for less time after repairs for two extra

[PHASE 4] CANDIDATE GENERATION

[PHASE 5] POSTPROCESSING

FINAL ANSWER:
Update laptop and other driver for less time after repairs for two extra

[PHASE 6] EVALUATION

GROUND TRUTH:
 Run hardware diagnostics and contact support for warranty replacement.

METRICS
Exact Match: 0
Token F1: 0.182
Semantic Overlap: 0.105
Aggregate Score: 0.096


QUESTION:
What is likely causing the internet issue?

[PHASE 1] QUERY PROCESSING

[PHASE 2] DOCUMENT PROCESSING

CONTEXT:
Customer reports internet disconnects frequently.Router firmware is outdated.Restart temporarily fixes issue.ISP outage was not

,Question,Predicted,Ground Truth,Exact Match,Token F1,Semantic Overlap,Final Score
0,What should the customer do next?,Update laptop and other driver for less time a...,Run hardware diagnostics and contact support ...,0,0.182,0.105,0.096
1,What is likely causing the internet issue?,Network Update (nutenability disorder * 1&#100...,The outdated router firmware is likely causing...,0,0.000,0.000,0.000
